# TATR-Span Ablation Study (E0~E3)

**실행 전 체크리스트**
- [ ] 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
- [ ] Google Drive에 `PubTables-1M-Structure/` 업로드 완료

| ID | 설명 | 핵심 옵션 |
|----|----|----|
| E0 | Baseline TATR | `--no_span_branch` |
| E1 | + Span Branch | ordinal regression head |
| E2 | + Hard Grid-Snap | curriculum warmup 5 epochs |
| E3 | + Soft Grid-Snap | curriculum warmup 5 epochs |

## 0. GPU 확인

In [ ]:
!nvidia-smi
import torch
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. 환경 설정

In [ ]:
!pip install -q pycocotools
print("Done")

In [ ]:
import os

REPO_DIR = "/content/t1"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/Jax0303/t1.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull
    print("Already cloned — pulled latest")

%cd {REPO_DIR}
!git log --oneline -3

## 2. 경로 설정

Drive에 올려야 할 폴더 구조:
```
MyDrive/PubTables-1M-Structure/
  images/              ← 94,959개 jpg  (필수, 21GB)
  train/               ← 86,284개 xml
  val/                 ← xml
  test/                ← xml
  train_filelist.txt
  val_filelist.txt
  test_filelist.txt
```
> **SUBSET_MODE = True** 로 먼저 돌려서 동작 확인 후 False로 전환 권장

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ── 경로 ───────────────────────────────────────────────────────
DATA_ROOT  = "/content/drive/MyDrive/PubTables-1M-Structure"
OUTPUT_DIR = "/content/outputs/ablation"

# tatr_base/src/main.py 는 CWD=tatr_base/src 에서 실행해야
# sys.path.append("../detr") 가 tatr_base/detr 를 올바르게 가리킴
SRC_DIR    = os.path.join(REPO_DIR, "tatr_base", "src")
CONFIG     = os.path.join(SRC_DIR, "structure_config.json")
TRAIN_PY   = os.path.join(SRC_DIR, "main.py")
EVAL_PY    = os.path.join(SRC_DIR, "eval_by_complexity.py")
AGG_PY     = os.path.join(REPO_DIR, "aggregate_results.py")

# ── 모드 ───────────────────────────────────────────────────────
# True  → 2000샘플 / 5 epoch smoke test  (실험 1회 ~15분)
# False → 전체 86K샘플 / 20 epoch        (실험 1회 ~3~4시간)
SUBSET_MODE = True
TRAIN_MAX   = 2000 if SUBSET_MODE else None
VAL_MAX     = 300  if SUBSET_MODE else None
EPOCHS      = 5    if SUBSET_MODE else 20
SEEDS       = [42, 43, 44]

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── 데이터 확인 ────────────────────────────────────────────────
for d in [DATA_ROOT,
          os.path.join(DATA_ROOT, 'images'),
          os.path.join(DATA_ROOT, 'train')]:
    exists = os.path.exists(d)
    count  = len(os.listdir(d)) if exists else 0
    print(f"{'OK' if exists else 'MISSING':<8} {d}  ({count:,} files)")

## 3. Sanity Check (CPU, 데이터 불필요)

In [ ]:
# 7/7 통과하면 학습 진행 OK
!python {REPO_DIR}/sanity_check.py

## 4. 학습 실행 (E0~E3)

각 셀이 독립적이므로 실험 단위로 개별 실행 가능.

In [ ]:
import subprocess, sys, os, time

def run_one(exp_id, seed, extra_args=()):
    """학습 1회 실행. CWD를 tatr_base/src 로 설정해야 sys.path가 올바르게 잡힘."""
    out_dir = os.path.join(OUTPUT_DIR, exp_id, f"seed{seed}")
    os.makedirs(out_dir, exist_ok=True)
    log_path = os.path.join(out_dir, "run.log")

    cmd = [
        sys.executable, TRAIN_PY,
        "--data_root_dir",  DATA_ROOT,
        "--config_file",    CONFIG,
        "--backbone",       "resnet18",
        "--data_type",      "structure",
        "--mode",           "train",
        "--epochs",         str(EPOCHS),
        "--model_save_dir", out_dir,
        "--metrics_save_filepath", os.path.join(out_dir, "metrics.json"),
        "--device",         "cuda",
        "--seed",           str(seed),
        "--num_workers",    "2",
    ]
    if TRAIN_MAX: cmd += ["--train_max_size", str(TRAIN_MAX)]
    if VAL_MAX:   cmd += ["--val_max_size",   str(VAL_MAX)]
    cmd += list(extra_args)

    print(f"\n{'='*55}")
    print(f"  {exp_id}  seed={seed}  epochs={EPOCHS}  subset={SUBSET_MODE}")
    print(f"  out: {out_dir}")
    print(f"{'='*55}")

    t0 = time.time()
    with open(log_path, 'w') as log:
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            cwd=SRC_DIR,   # ← 핵심: ../detr 경로가 tatr_base/detr 를 가리키도록
        )
        for line in proc.stdout:
            print(line, end='')  # 실시간 출력
            log.write(line)
        proc.wait()

    elapsed = (time.time() - t0) / 60
    ok = proc.returncode == 0
    print(f"\n{'OK' if ok else 'FAIL'} {exp_id} seed={seed} — {elapsed:.1f}분  (rc={proc.returncode})")
    return ok

print("run_one() ready")

In [ ]:
# ── E0: Baseline TATR ──────────────────────────────────────────
for seed in SEEDS:
    run_one("E0_baseline", seed, extra_args=(
        "--no_span_branch",
        "--no_grid_snapping",
    ))

In [ ]:
# ── E1: + Span Attribute Branch ────────────────────────────────
for seed in SEEDS:
    run_one("E1_span", seed, extra_args=(
        "--span_loss_coef", "0.5",
        "--no_grid_snapping",
    ))

In [ ]:
# ── E2: + Hard Grid-Snapping ───────────────────────────────────
for seed in SEEDS:
    run_one("E2_snap_hard", seed, extra_args=(
        "--span_loss_coef", "0.5",
        "--grid_snapping",  "hard",
        "--n_warm",         "5",
    ))

In [ ]:
# ── E3: + Soft Grid-Snapping ───────────────────────────────────
for seed in SEEDS:
    run_one("E3_snap_soft", seed, extra_args=(
        "--span_loss_coef", "0.5",
        "--grid_snapping",  "soft",
        "--n_warm",         "5",
    ))

## 5. 복잡도별 평가

In [ ]:
import subprocess, sys

out_csv = os.path.join(OUTPUT_DIR, "results_by_complexity.csv")

proc = subprocess.run(
    [sys.executable, EVAL_PY,
     "--results_dir", OUTPUT_DIR,
     "--data_root",   DATA_ROOT,
     "--xml_subdir",  "test",
     "--output_csv",  out_csv],
    cwd=SRC_DIR,   # ← 여기도 동일하게 SRC_DIR 기준
    capture_output=False,
)
print(f"\nrc={proc.returncode}  →  {out_csv}")

## 6. 결과 집계 (Table A / Table B)

In [ ]:
import subprocess, sys

proc = subprocess.run(
    [sys.executable, AGG_PY,
     "--results_dir", OUTPUT_DIR,
     "--output_csv",  os.path.join(OUTPUT_DIR, "summary_results.csv")],
    cwd=REPO_DIR,  # aggregate_results.py 는 repo root 기준
    capture_output=False,
)
print(f"rc={proc.returncode}")

## 7. 결과 시각화

In [ ]:
import pandas as pd

csv_path = os.path.join(OUTPUT_DIR, "results_by_complexity.csv")
df = pd.read_csv(csv_path)

# 전체(all) 요약
summary = (
    df[df['split'] == 'all']
    .groupby('experiment_id')[['GriTS_Top', 'GriTS_Loc', 'GriTS_Con']]
    .agg(['mean', 'std'])
    .round(4)
)
display(summary)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

exps   = ['E0_baseline', 'E1_span', 'E2_snap_hard', 'E3_snap_soft']
labels = ['E0\nBaseline', 'E1\nSpan', 'E2\nHard', 'E3\nSoft']
colors = {'simple': '#4C72B0', 'complex': '#DD8452'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, metric in zip(axes, ['GriTS_Loc', 'GriTS_Top']):
    for split, offset in [('simple', -0.2), ('complex', 0.2)]:
        sub = df[df['split'] == split].copy()
        sub[metric] = sub[metric].astype(float)
        means = [sub[sub['experiment_id']==e][metric].mean() for e in exps]
        stds  = [sub[sub['experiment_id']==e][metric].std()  for e in exps]
        ax.bar(np.arange(len(exps)) + offset, means, 0.35,
               yerr=stds, label=split, color=colors[split], capsize=4, alpha=0.85)
    ax.set_title(metric); ax.set_xticks(range(len(exps))); ax.set_xticklabels(labels)
    ax.set_ylim(0, 1); ax.legend(); ax.grid(axis='y', alpha=0.3)

fig.suptitle('Simple vs Complex (mean ± std, 3 seeds)')
plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'ablation_main.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show(); print(f"Saved: {save_path}")

In [ ]:
kbins = ['k=1', 'k=2', 'k=3~4', 'k>=5']
e0_m, e3_m = [], []
for kb in kbins:
    s0 = df[(df['experiment_id']=='E0_baseline') & (df['k_bin']==kb)]['GriTS_Loc'].astype(float)
    s3 = df[(df['experiment_id']=='E3_snap_soft') & (df['k_bin']==kb)]['GriTS_Loc'].astype(float)
    e0_m.append(s0.mean() if len(s0) else float('nan'))
    e3_m.append(s3.mean() if len(s3) else float('nan'))

x = np.arange(len(kbins))
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x-0.2, e0_m, 0.35, label='E0 Baseline', color='#4C72B0', alpha=0.85)
ax.bar(x+0.2, e3_m, 0.35, label='E3 Soft-Snap', color='#55A868', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(kbins)
ax.set_ylabel('GriTS_Loc'); ax.set_ylim(0, 1)
ax.set_title('GriTS_Loc by Span Complexity: E0 vs E3')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'ablation_kbin.png')
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show(); print(f"Saved: {save_path}")

## 8. 결과 Drive 백업

In [ ]:
import shutil, time

timestamp  = time.strftime('%Y%m%d_%H%M')
backup_dst = f"/content/drive/MyDrive/ablation_results_{timestamp}"
shutil.copytree(OUTPUT_DIR, backup_dst)
print(f"Backed up → {backup_dst}")

## 9. 세션 끊긴 후 이어받기

Colab 세션 초기화 시: 섹션 1~2 먼저 실행 후 아래 셀로 특정 실험만 재시작.

In [ ]:
RESUME_EXP  = "E3_snap_soft"  # 이어받을 실험 ID
RESUME_SEED = 42
LOAD_PATH   = os.path.join(OUTPUT_DIR, RESUME_EXP, f"seed{RESUME_SEED}", "model.pth")

if os.path.exists(LOAD_PATH):
    print(f"체크포인트 발견: {LOAD_PATH}")
    run_one(RESUME_EXP, RESUME_SEED,
            extra_args=(
                "--span_loss_coef", "0.5",
                "--grid_snapping",  "soft",
                "--n_warm",         "5",
                "--model_load_path", LOAD_PATH,
            ))
else:
    print(f"체크포인트 없음: {LOAD_PATH}")
    print("Drive 백업에서 /content/outputs/ablation/ 으로 복원 후 재시도")

## 부록. 전체 학습 시간 가이드

| 모드 | 샘플 | Epochs | 실험 1회 | 전체 12회 |
|------|------|--------|---------|----------|
| `SUBSET_MODE=True`  | 2,000 | 5  | ~15분    | ~3시간    |
| `SUBSET_MODE=False` | 86K  | 20 | ~3~4시간 | ~36~48시간 |

- **T4 무료**: 세션당 최대 12시간 → E0/E1 먼저, E2/E3 다음 세션
- **Colab Pro+ A100**: 전체 12회를 한 세션에 완주 가능
- 세션 끊기기 전 반드시 섹션 8(Drive 백업) 실행